# Analysis daily data Innsbruck 1925 - 2025

- `rr` ...Niederschlag 24h Summe,"Niederschlag, 24-Stundensumme der Einminutenwerte von 06:01 UTC bis 06:00 UTC des Folgetags, -1=kein Niederschlag, 0=weniger als 1/10 mm",mm,
- `so_h` ...Sonnenscheindauer,"Sonnenscheindauer, Summe aus den Stundenwerten 0-24 Uhr MOZ (23 Vortag - 23 Tag UTC)",h,
- `tlmax` ...Lufttemperatur 2m Maximalwert,"Lufttemperatur in 2m Höhe, Maximum der Zehnminutenwerte (tlmax) von 18:10 UTC Vortag - 18:00 UTC aktueller Tag",°C,
- `tlmin` ...Lufttemperatur 2m Minimalwert,"Lufttemperatur in 2m Höhe, Minimum der Zehnminutenwerte (tlmin) von 18:10 UTC Vortag - 18:00 UTC aktueller Tag",°C,
- `tl_mittel` ...Lufttemperatur 2m Mittelwert,"Lufttemperatur in 2m Höhe, Mittelwert berechnet aus (tlmax + tlmin)/2",°C,
- `rf_mittel` ...Relative Feuchte Mittelwert,"Relative Feuchte, Tagesmittel aus den Beobachtungsterminen I,II,III (06,13,18 UTC)",%,


## Rain categories

- `Very Light Rain` ...Rainfall amount realised in a day is between 0.1 to 0.9 mm
- `Light Rain` ...Rainfall amount realised in a day is between 1.0 mm to 10 mm
- `Moderate Rain` ...Rainfall amount realised in a day is between 11 to 30 mm
- `Heavy Rain` ...Rainfall amount realised in a day is between 31.0 to 70.0 mm
- `Very Heavy Rain` ...Rainfall amount realised in a day is between 71.0 to 150 mm
- `Extremely Heavy Rain` ...Rainfall amount realised in a day is equal or more than 151 mm
- `Rainy Day` ...Rainfall amount realised in a day is 1.0 mm or more

In [ ]:
# Climate_Change_Analysis.ipynb

# Imports Libs
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression

# Import Data
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
file_name = "Messstationen Tagesdaten v2 Datensatz_19250101_20250817.csv"
relative_path = os.path.join("..", "data", file_name)
csv_path = os.path.abspath(relative_path)

df = pd.read_csv(csv_path, parse_dates=["time"])

# Step 3: Preprocess
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day

# Check for missing values
print(df.isnull().sum())
display(df.head(5))
display(df.tail(5))


## Rain statistics

In [ ]:
# Data wrengling

# Replace -1 with NaN (no data), and 0 with 0.0 (trace rain)
df["rr"] = df["rr"].astype("float").replace(-1.0, pd.NA)


In [ ]:
# Check totalrian/year  how many days it rained (rr > 10 mm)

rain_days = df[df["rr"] > 10.0].groupby("year").size().rename("rain_day_count")
total_rain = df.groupby("year")["rr"].sum().rename("total_rain_mm")

rain_stats = pd.concat([rain_days, total_rain], axis=1)
rain_stats["avg_rain_per_day"] = rain_stats["total_rain_mm"] / rain_stats["rain_day_count"]


# Per Month
# jul_2025 = df[(df["year"] == 2025) & (df["month"] == 7)]
# rain_days_jul_2025 = jul_2025[jul_2025["rr"] > 10.0].shape[0]

# display(rain_days_jul_2025)
# display(jul_2025)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(rain_stats.index, rain_stats["rain_day_count"], label="Rain Days")
plt.title("Rain Days")
plt.xlabel("Year")
plt.ylabel("Rain days")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(12, 6))
plt.plot(rain_stats.index, rain_stats["total_rain_mm"], label="Total Rainfall (mm)")
plt.title("Total Rainfall")
plt.xlabel("Year")
plt.ylabel("Millimeters")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))

plt.plot(rain_stats.index, rain_stats["avg_rain_per_day"], label="Avg Rain per Rainy Day (mm)", linestyle="--")
plt.title("Avg Rain per Rainy Day")
plt.xlabel("Year")
plt.ylabel("Millimeters")
plt.legend()
plt.grid(True)
plt.show()


# Correlation Matrix
# Select relevant numeric columns
corr_matrix = rain_stats.corr()

# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Correlation Matrix of Rain Statistics")
plt.tight_layout()
plt.show()


## Temperature 

- Correlation Temp - sun hours
- temp average year
- temp min year
- temp max year

In [ ]:
# Group by year and calculate correlation between so_h and tl_mittel
yearly_corr = df.groupby("year").apply(lambda x: x["so_h"].corr(x["tl_mittel"])).rename("sun_temp_corr")

# Plot the correlation over time
plt.figure(figsize=(12, 6))
yearly_corr.plot()
plt.title("Yearly Correlation Between Sun Hours and Mean Temperature (1925–2025)")
plt.xlabel("Year")
plt.ylabel("Correlation Coefficient")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
df_year = df.groupby("year").mean(numeric_only=True).reset_index()
df_temp_year = df_year["tl_mittel"]

# # Plot
# plt.figure(figsize=(12, 6))
# df_temp_year.plot(color="tomato", linewidth=2)
# plt.title("Yearly Average Temperature (1925–2025)")
# plt.xlabel("Year")
# plt.ylabel("Mean Temperature (°C)")
# plt.grid(True)
# plt.tight_layout()
# plt.show()

# Linear Trend Analysis

# Prepare data
X = df_year["year"].values.reshape(-1, 1)
y_temp = df_year["tl_mittel"].values

# Fit linear regression model
model_temp = LinearRegression().fit(X, y_temp)
trend_temp = model_temp.predict(X)

# Plot
plt.figure(figsize=(12, 6))
plt.plot(df_year["year"], y_temp, label="Average Temperature", color="orange")
plt.plot(df_year["year"], trend_temp, label="Linear Trend", linestyle="--", color="red")
plt.title("Linear Trend in Average Temperature (1925–2025)")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Print slope and R²
print(f"Slope: {model_temp.coef_[0]:.4f} °C/year")
print(f"R²: {model_temp.score(X, y_temp):.4f}")


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# Prepare data
X = df_year["year"].values.reshape(-1, 1)
y = df_year["tl_mittel"].values

# Create polynomial features (degree 3 for cubic)
poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)

# Fit the model
model = LinearRegression().fit(X_poly, y)
y_pred = model.predict(X_poly)

# Print model quality
r_squared = model.score(X_poly, y)
print(f"Cubic Regression R²: {r_squared:.4f}")
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

# Plot
plt.figure(figsize=(12, 6))
plt.scatter(df_year["year"], y, label="Actual Temperature", color="blue")
plt.plot(df_year["year"], y_pred, label="Cubic Trend", color="red", linewidth=2)
plt.title("Cubic Regression of Average Temperature Over Years")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
